# C11-neural-training — Practice p19 — Solution


**Type:** scenario analysis · **Difficulty:** advanced · **Concepts:** autograd-training, torch-optimizers


Iteration 1 reaches `step` with every gradient `None`, so SGD neither moves a
parameter nor creates momentum state. `backward` then creates finite gradients,
which `zero_grad(set_to_none=True)` immediately deletes. Iteration 2 repeats the
same state transitions. Different batches can change the reported loss even
though the model is fixed, so loss variation alone is not training evidence.
Wrapping `step` in `no_grad` cannot help: optimizer steps already perform
updates without building an autograd graph, and the required gradients are
still absent at that earlier line.

The correct lifecycle is:
```python
for xb, yb in loader:
    optimizer.zero_grad(set_to_none=True)
    prediction = model(xb)
    loss = ((prediction - yb) ** 2).mean()
    loss.backward()
    optimizer.step()
```
Immediately after backward, certify every optimizer-owned gradient is non-None,
finite, and shape-equal to its parameter. After step, compare parameters with
pre-step clones and require nonzero movement plus populated momentum state.
Removing `backward()` fails the gradient certificate; replacing `step()` by a
no-op fails movement and momentum-state certificates.


In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(19); torch.set_default_dtype(torch.float64)
model_p19=nn.Linear(2,1); optimizer_p19=torch.optim.SGD(model_p19.parameters(),lr=.05,momentum=.9)
X_p19=torch.tensor([[1.,2.],[-1.,1.]]); y_p19=torch.tensor([[1.],[0.]])
optimizer_p19.zero_grad(set_to_none=True); before_p19=[p.detach().clone() for p in model_p19.parameters()]
loss_p19=((model_p19(X_p19)-y_p19)**2).mean(); loss_p19.backward()
grad_cert_p19=all(p.grad is not None and p.grad.shape==p.shape and torch.isfinite(p.grad).all() for p in model_p19.parameters())
optimizer_p19.step(); movement_p19=max(float(torch.linalg.vector_norm(p-q)) for p,q in zip(model_p19.parameters(),before_p19))


### Answer check


In [ ]:
assert grad_cert_p19
assert movement_p19>0.0
assert len(optimizer_p19.state)==len(list(model_p19.parameters()))
assert all("momentum_buffer" in state for state in optimizer_p19.state.values())
